### Description

In [0]:
# ---------------------------------------------------------------------------
# Cohort-based DB vs S3 validation: high-level, commented summary
# ---------------------------------------------------------------------------
# Reference section:

# ---------------------------------------------------------------------------
# 1. Defines environment and constants
# ---------------------------------------------------------------------------
# - Sets catalog and schema names, e.g.:
#     CATALOG = "datascience_ea_dev"
#     SCHEMA  = "pe_slv"
#     SCHEMA_MEMBER_DNA = "pe"
#   and ALL_SCHEMAS = [SCHEMA, SCHEMA_MEMBER_DNA].
#
# - DETAIL points to a key transaction detail table used as a bridge:
#     DETAIL = f"{CATALOG}.{SCHEMA}.transaction_fiscal_detail"
#
# - N_MEMBERS = 1000 controls the size of the member cohort sample.
#
# - S3_FILES is a Python dict mapping logical table names to S3 paths, e.g.:
#     S3_FILES = {
#         "master_member": "s3://bucket/path/master_member.parquet",
#         "transaction_fiscal_detail": "s3://bucket/path/transaction_fiscal_detail.parquet",
#         ...
#     }

# ---------------------------------------------------------------------------
# 2. Reads a member cohort from S3 (master_member)
# ---------------------------------------------------------------------------
# - Reads the S3 file for "master_member" (CSV or Parquet) using a helper:
#     read_s3_any(path) -> DataFrame
#   which:
#     * infers schema, handles header for CSV
#     * or reads Parquet directly.
#
# - Normalizes column names to UPPER case with:
#     normalize_cols_upper(df).
#
# - From master_member, it:
#     * selects MBRSHP_SID
#     * drops NULL MBRSHP_SIDs
#     * sorts by MBRSHP_SID
#     * limits to N_MEMBERS
#     * drops duplicates on MBRSHP_SID
#
# - This result is:
#     selected_members
#   and is registered as a temp view:
#     selected_members
#
# - A broadcasted DataFrame of just the distinct MBRSHP_SID values is created:
#     cohort_sids = broadcast(selected_members.select("MBRSHP_SID").distinct())

# ---------------------------------------------------------------------------
# 3. Discovers DB schema and builds a column dictionary
# ---------------------------------------------------------------------------
# - Queries the information_schema for all columns in ALL_SCHEMAS:
#     SELECT table_schema, table_name, column_name, data_type
#     FROM   {CATALOG}.information_schema.columns
#     WHERE  table_schema IN ('pe_slv', 'pe')
#
# - Stores the result in cols_df_full, then:
#     * cols_df = cols_df_full.select("table_name", "column_name", "data_type")
#       (schema-agnostic)
#
# - Builds cols_by_table, a Python dict:
#     cols_by_table[(schema_name, table_name)] = [
#         {"column_name": ..., "data_type": ...}, ...
#     ]
#   This is used later to infer how tables link to members and to categorize
#   columns by type (date/numeric/string).

# ---------------------------------------------------------------------------
# 4. Infers a “link plan” for how to join tables to the member cohort
# ---------------------------------------------------------------------------
# - For each (schema, table) in cols_by_table, inspects what key columns exist:
#     * MBRSHP_SID
#     * MBRSHP_NBR
#     * PURCH_HDR_ID
#     * PURCH_DTL_ID
#
# - Based on presence of these columns (and DETAIL’s existence), chooses
#   a linking strategy and join_key:
#     * "direct_sid"     if table has MBRSHP_SID          (join_key = "MBRSHP_SID")
#     * "direct_nbr"     if table has MBRSHP_NBR          (join_key = "MBRSHP_NBR")
#     * "via_purch_hdr"  if table has PURCH_HDR_ID        (join_key = "PURCH_HDR_ID")
#     * "via_purch_dtl"  if table has PURCH_DTL_ID        (join_key = "PURCH_DTL_ID")
#
# - Produces link_plan_df with columns:
#     table_schema, table_name, strategy, join_key
#
# - Then filters link_plan_df to only tables whose table_name is in S3_FILES
#   (i.e., there is an S3 file to compare against).

# ---------------------------------------------------------------------------
# 5. Column type utilities
# ---------------------------------------------------------------------------
# - Using cols_by_table, the code categorizes columns per table name into:
#     * date_cols   (DATE, TIMESTAMP)
#     * num_cols    (INTEGER, BIGINT, DOUBLE, DECIMAL, NUMERIC, etc.)
#     * string_cols (STRING, CHAR, VARCHAR)
#
# - The main helper:
#     cols_by_type_from_dict(table_name) -> (date_cols, num_cols, string_cols)
#
# - infer_categorical_cols(df, string_cols, max_cardinality=100):
#     * For each candidate string column, counts distinct values up to
#       max_cardinality + 1.
#     * Marks columns with ≤ max_cardinality distinct values as “categorical”.
#
# - needed_cols_for_table(schema_name, tname, strategy, join_key):
#     * Computes the minimal set of columns to read:
#       join columns + date_cols + num_cols + string_cols
#     * (In the provided snippet, this helper is defined but not used later.)

# ---------------------------------------------------------------------------
# 6. Metric computation and comparison
# ---------------------------------------------------------------------------
# - compute_metrics(df, date_cols, num_cols):
#     * rec["row_count"] = df.count()
#     * For each date column c:
#         - rec[f"date_min__{c}"] = min(c)
#         - rec[f"date_max__{c}"] = max(c)
#     * For each numeric column c:
#         - rec[f"num_sum__{c}"] = sum(c)
#         - rec[f"num_avg__{c}"] = avg(c)
#         - rec[f"num_std__{c}"] = stddev(c)
#     * Returns a flat dict like:
#         {
#           "row_count": 123,
#           "date_min__ORDER_DATE": "...",
#           "num_sum__AMOUNT": 456.78,
#           ...
#         }
#
# - compare_metrics(db_rec, s3_rec, date_cols, num_cols):
#     * Compares DB metrics vs S3 metrics:
#         - row_count
#         - date_min/date_max (as strings)
#         - numeric sum/avg/std with tolerances:
#             tol_abs  = 1e-6
#             tol_ratio = 0.01 (1% relative difference)
#     * If any differences exceed tolerance (for numerics) or differ
#       (for dates and row_count), records an issue message.
#     * Returns:
#         verdict = "OK" or "MISMATCH"
#         issues  = list of mismatch descriptions.
#
# - to_maps(rec, date_cols, num_cols):
#     * Converts the flat metrics dict into several per-column maps:
#         - date_min: { col -> min_value_as_string }
#         - date_max: { col -> max_value_as_string }
#         - num_sum:  { col -> sum_float }
#         - num_avg:  { col -> avg_float }
#         - num_std:  { col -> std_float }

# ---------------------------------------------------------------------------
# 7. Builds S3-side cohort slices (s3_cohort_df)
# ---------------------------------------------------------------------------
# - s3_cohort_df(table_name, strategy, join_key) returns:
#     (cohort_sliced_df, None) on success
#     (None, "error message") on failure.
#
# - Steps:
#     * Reads the S3 file for the given table_name (using S3_FILES[table_name]).
#     * Normalizes columns to UPPER case.
#     * Then, depending on strategy, filters to the same member cohort:
#
#   - Strategy: "direct_sid"
#       * Requires MBRSHP_SID in the S3 table.
#       * df = base.join(cohort_sids, on="MBRSHP_SID", how="inner").
#
#   - Strategy: "direct_nbr"
#       * Requires S3 master_member path.
#       * Builds a bridge from master_member: (MBRSHP_NBR → MBRSHP_SID).
#       * df = base
#             .join(broadcast(master_member_bridge), on="MBRSHP_NBR", "inner")
#             .join(cohort_sids, on="MBRSHP_SID", "inner").
#
#   - Strategy: "via_purch_hdr"
#       * Requires S3 transaction_fiscal_detail path.
#       * Uses transaction_fiscal_detail as a bridge:
#         PURCH_HDR_ID → MBRSHP_SID.
#       * df = base
#             .join(broadcast(detail_bridge), on="PURCH_HDR_ID", "inner")
#             .join(cohort_sids, on="MBRSHP_SID", "inner").
#
#   - Strategy: "via_purch_dtl"
#       * Same pattern, but via PURCH_DTL_ID → MBRSHP_SID.
#
# - If required columns or S3 paths are missing, returns an error string.
#   These errors are later recorded as SKIPPED_S3 for that table.

# ---------------------------------------------------------------------------
# 8. Builds DB-side cohort slices (build_db_cohort_slice)
# ---------------------------------------------------------------------------
# - build_db_cohort_slice(schema_name, tname, strat, key) returns:
#     (db_df, join_path_str, None) on success
#     (None, None, "STATUS_CODE: message") on failure.
#
# - For each table in the link plan:
#     * Checks if the Databricks table exists:
#         table_fq = f"{CATALOG}.{schema_name}.{tname}"
#         table_exists(table_fq)
#
#   - Strategy: "direct_sid"
#       * Requires MBRSHP_SID in DB table.
#       * db_df = base_df.join(selected_members, on="MBRSHP_SID", how="inner").
#       * join_path describes this join in human-readable form.
#
#   - Strategy: "direct_nbr"
#       * Uses S3 master_member as a bridge from MBRSHP_NBR → MBRSHP_SID,
#         then joins to selected_members.
#       * Requires MBRSHP_NBR in DB table and MBRSHP_NBR/MBRSHP_SID
#         in S3 master_member.
#
#   - Strategy: "via_purch_hdr"
#       * Uses DB DETAIL table (transaction_fiscal_detail) as a bridge:
#         PURCH_HDR_ID → MBRSHP_SID → selected_members.
#
#   - Strategy: "via_purch_dtl"
#       * Similar to via_purch_hdr, but with PURCH_DTL_ID.
#
# - On problems (e.g., table missing, bridge column missing), returns
#   specific status codes:
#     * "SKIPPED_DB_TABLE_MISSING: ..."
#     * "SKIPPED_DB_BRIDGE_MISSING: ..."
#     * "SKIPPED_UNKNOWN_STRATEGY: ..."

# ---------------------------------------------------------------------------
# 9. Main loop: compare each table DB vs S3
# ---------------------------------------------------------------------------
# - Iterates over link_plan_df.rows ordered by (strategy, table_schema, table_name).
#
# For each row (schema_name, tname, strat, key):
#
#   1) Build DB slice:
#       * db_df, join_path, db_err = build_db_cohort_slice(...)
#       * If db_err is not None:
#           - Parses error into status + message.
#           - Appends a mostly-empty result via make_empty_result():
#               - status = e.g. "SKIPPED_DB_TABLE_MISSING"
#               - issues = [message]
#               - db_row_count = db_df.count() if db_df else 0
#           - Continues to next table.
#
#   2) Build S3 slice:
#       * s3_df, s3_err = s3_cohort_df(tname, strat, key)
#       * If s3_err is not None:
#           - Appends a result with:
#               status = "SKIPPED_S3"
#               issues = [s3_err]
#               db_row_count = db_df.count()
#               s3_row_count = 0
#           - Continues to next table.
#
#   3) Debug temp views:
#       * Writes:
#           dbg_{schema_name}_{tname} for DB slice
#           s3g_{schema_name}_{tname} for S3 slice
#         so you can inspect intermediate results in SQL.
#
#   4) Align columns & infer types:
#       * date_cols, num_cols, string_cols = cols_by_type_from_dict(tname).
#       * Filters these lists to only keep columns present in both DB and S3:
#           date_cols   = [c for c in date_cols   if c in db_df and c in s3_df]
#           num_cols    = [c for c in num_cols    if c in db_df and c in s3_df]
#           string_cols = [c for c in string_cols if c in db_df and c in s3_df]
#
#       * infers categorical columns separately on DB and S3 side, then
#         takes intersection, giving categorical_cols.
#
#   5) Compute metrics & compare:
#       * db_rec = compute_metrics(db_df, date_cols, num_cols)
#       * s3_rec = compute_metrics(s3_df, date_cols, num_cols)
#       * verdict, issues = compare_metrics(db_rec, s3_rec, date_cols, num_cols)
#
#       * Converts metrics dicts to maps via:
#           db_date_min, db_date_max, db_sum, db_avg, db_std = to_maps(db_rec, ...)
#           s3_date_min, s3_date_max, s3_sum, s3_avg, s3_std = to_maps(s3_rec, ...)
#
#   6) Append a result record:
#       * For each table, adds:
#           - table_name  (e.g. "pe_slv.some_table")
#           - strategy, join_key, join_path
#           - status = verdict ("OK"/"MISMATCH") or a SKIPPED_* code
#           - issues = list of descriptive strings
#           - db_row_count, s3_row_count
#           - date_cols, numeric_cols, string_cols, categorical_cols
#           - db_date_min, db_date_max, s3_date_min, s3_date_max
#           - db_sum, db_avg, db_std, s3_sum, s3_avg, s3_std

# ---------------------------------------------------------------------------
# 10. Builds a final validation DataFrame
# ---------------------------------------------------------------------------
# - Defines an explicit schema for the validation output with fields:
#     table_name, strategy, join_key, join_path,
#     status, issues,
#     db_row_count, s3_row_count,
#     date_cols, numeric_cols, string_cols, categorical_cols,
#     db_date_min, db_date_max, s3_date_min, s3_date_max,
#     db_sum, db_avg, db_std, s3_sum, s3_avg, s3_std
#
# - Creates:
#     validation_table = spark.createDataFrame(rows, schema=schema)
#
# - Registers it as temp view:
#     cohort_validation_db_vs_s3
#
# - Finally, displays results ordered by:
#     status ascending, db_row_count descending
#   which highlights mismatches and large tables first.

# ---------------------------------------------------------------------------
# In plain terms:
# ---------------------------------------------------------------------------
# - Automatically discovers how each relevant DB table can be joined
#   back to a fixed sample of members (the cohort).
#
# - For each table that also exists in S3, extracts matching slices
#   from Databricks (DB) and S3 for the same members.
#
# - Computes summary metrics on both sides:
#     row counts, date ranges, numeric sums/averages/stddevs.
#
# - Compares these metrics and records where DB and S3 disagree.
#
# - Produces a consolidated validation table with one row per table,
#   summarizing:
#     * join strategy and join path
#     * status ("OK", "MISMATCH", "SKIPPED_*")
#     * mismatch details (if any)
#     * metrics and column lists, for both DB and S3.

In [0]:
s3_files = {
    'bcg_maps_ah5_customer_facing_desc': 's3://memberanalytics-data-out-prod/pipelined_intermediates/bcg_maps/AH5_custumer_facing_desc/AH5_custumer_facing_desc.csv',
    'awards': 's3://memberanalytics-data-out-prod/pipelined_intermediates/awards',
    'awards_fiscal': 's3://memberanalytics-data-out-prod/pipelined_intermediates/awards_fiscal',
    'master_brand': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/brand',
    'master_census_tract': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/census_tract',
    'master_club_with_brand': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/club_with_brand',
    'master_club_square_with_brand': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/club_square_with_brand',
    'coupon_clip': 's3://memberanalytics-data-out-prod/pipelined_intermediates/coupon_clip',
    'coupon_clip_fiscal': 's3://memberanalytics-data-out-prod/pipelined_intermediates/coupon_clip_fiscal',
    'transaction_fiscal_detail': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction_fiscal/detail',
    'transaction_fiscal_detail_gas_nr': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction_fiscal/detail_gas_nr',
    'transaction_fiscal_detail_isnr': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction_fiscal/detail_isnr',
    'master_email': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/email',
    'master_email_fiscal': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/email_fiscal',
    'exclusions_brand_exclusions_mixed': 's3://memberanalytics-data-out-prod/exclusions/input/brand_exclusions_mixed.csv',
    'lookup_fiscal_days': 's3://memberanalytics-data-out-prod/pipelined_intermediates/lookup/fiscal_days',
    'transaction_header': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction/header',
    'transaction_fiscal_header': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction_fiscal/header',
    'master_item': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/item',
    'ad_hoc_master_item_with_brand': 's3://memberanalytics-data-out-prod/AD_HOC/pipelined_intermediates/master/item_with_brand',
    'master_member': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/member',
    'master_member_extended': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/member_extended',
    'master_member_history': 's3://memberanalytics-data-out-prod/pipelined_intermediates/master/member_history',
    'transaction_payment': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction/payment',
    'transaction_fiscal_payment': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction_fiscal/payment',
    'quotient_id': 's3://memberanalytics-data-out-prod/pipelined_intermediates/quotient_id',
    'bcg_maps_strategic_segments': 's3://memberanalytics-data-out-prod/pipelined_intermediates/bcg_maps/strategic_segment/Strategic_Segments.csv',
    'skeleton': 's3://memberanalytics-data-out-prod/pipelined_intermediates/skeleton',
    'control_files_tab_01_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_01_comparison.csv',
    'control_files_tab_02_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_02_comparison.csv',
    'control_files_tab_03_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_03_comparison.csv',
    'control_files_tab_04_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_04_comparison.csv',
    'control_files_tab_05_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_05_comparison.csv',
    'control_files_tab_06_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_06_comparison.csv',
    'control_files_tab_07_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_07_comparison.csv',
    'control_files_tab_08_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_08_comparison.csv',
    'control_files_tab_09_comparison': 's3://memberanalytics-data-out-prod/pipelined_intermediates/control_files//tab_09_comparison.csv',
    'bcg_maps_tender_type_group_csv': 's3://memberanalytics-data-out-prod/pipelined_intermediates/bcg_maps/tender_type_group_csv',
    'transaction_detail': 's3://memberanalytics-data-out-prod/pipelined_intermediates/transaction/detail',
    'customer_cube_full': 's3://memberanalytics-data-out-prod/CUBES/customer_cube_full/customer_cube',
    'ah4_cd_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/AH4_CD/CATEGORY_DNA_full/PARQUET',
    'ah5_cd_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/AH5_CD/CATEGORY_DNA_full/PARQUET',
    'article_nbr_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/ARTICLE_NBR/CATEGORY_DNA_full/PARQUET',
    'brand_cd_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/BRAND_CD/CATEGORY_DNA_full/PARQUET',
    'fs_customer_cube_full': 's3://memberanalytics-data-out-prod/CUBES/customer_cube_full/customer_cube',
    'fs_ah4_cd_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/AH4_CD/CATEGORY_DNA_full/PARQUET',
    'fs_ah5_cd_category_dna_full':'s3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/AH5_CD/CATEGORY_DNA_full/PARQUET',
    'fs_article_nbr_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/ARTICLE_NBR/CATEGORY_DNA_full/PARQUET',
    'fs_brand_cd_category_dna_full': 's3://memberanalytics-data-out-prod/CATEGORY_DNA/PROD/BRAND_CD/CATEGORY_DNA_full/PARQUET'
}

### ETL & MEMBER DNA

In [0]:

# === Setup used by the comparison block below (serverless-safe) ===
from pyspark.sql import functions as F, types as T
from pyspark.sql.utils import AnalysisException

# -----------------------------
# Config 
# -----------------------------
CATALOG = "datascience_ea_dev"
SCHEMA  = "pe_slv"          # main schema (silver)
SCHEMA_MEMBER_DNA = "pe"    # DNA schema
ALL_SCHEMAS   = [SCHEMA, SCHEMA_MEMBER_DNA]

DETAIL  = f"{CATALOG}.{SCHEMA}.transaction_fiscal_detail"
N_MEMBERS = 1_000  # _000

# Provide your S3 path map here (must already be defined or paste your dict)
S3_FILES = s3_files  


In [0]:
print(SCHEMA)
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(100, truncate=False)

print(SCHEMA_MEMBER_DNA)
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_MEMBER_DNA}").show(100, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# Cohort = a fixed subset of members used to compare datasets.
# Link plan = rules that say, for each table, how to reach that cohort (which key/strategy).
# Code = applies link plan to DB and S3, computes metrics to check matches

# Define a member cohort (a fixed list of member IDs) from an S3 file.
# Inspect the database schema to understand what columns each table has.
# Infer a “link plan” describing how each table can be joined back to that member cohort (by SID, by member number, or via purchase header/detail).
# Prepare column metadata and linking rules that can then be used to compute metrics comparing S3 data to database tables.

CATALOG = "datascience_ea_dev"
SCHEMA  = "pe_slv"
SCHEMA_MEMBER_DNA = "pe"
ALL_SCHEMAS = [SCHEMA, SCHEMA_MEMBER_DNA]

DETAIL  = f"{CATALOG}.{SCHEMA}.transaction_fiscal_detail"
N_MEMBERS = 1_000

S3_FILES = s3_files

def read_s3_any(path: str):
    p = path.lower()
    if p.endswith(".csv"):
        return (spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(path))
    else:
        return spark.read.parquet(path)

def normalize_cols_upper(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.upper())
    return df

def table_exists(fq_name: str) -> bool:
    try:
        spark.table(fq_name).limit(1).collect()
        return True
    except Exception:
        return False

# MBRSHP_SID
# MBRSHP_NBR
# PURCH_HDR_ID
# PURCH_DTL_ID
def needed_cols_for_table(schema_name, tname, strategy, join_key):
    """
    Return the minimal set of columns to read from the table:
    - join keys (based on strategy)
    - date + numeric + string columns used for metrics / categorical inference
    """
    date_cols, num_cols, string_cols = cols_by_type_from_dict(tname)

    # join columns needed per strategy
    join_cols = []
    if strategy == "direct_sid":
        join_cols = ["MBRSHP_SID"]
    elif strategy == "direct_nbr":
        join_cols = ["MBRSHP_NBR"]
    elif strategy == "via_purch_hdr":
        join_cols = ["PURCH_HDR_ID"]
    elif strategy == "via_purch_dtl":
        join_cols = ["PURCH_DTL_ID"]

    # We only need join columns + metric columns + string cols
    needed = sorted(set(join_cols + date_cols + num_cols + string_cols))
    return needed

# -----------------------------
# Cohort from S3 master_member -> selected_members, cohort_sids
# -----------------------------
if "master_member" not in S3_FILES:
    raise RuntimeError("S3 path for 'master_member' is missing in s3_files.")

selected_members = (
    normalize_cols_upper(read_s3_any(S3_FILES["master_member"]))
        .select("MBRSHP_SID")
        .where(F.col("MBRSHP_SID").isNotNull())
        .orderBy("MBRSHP_SID")
        .limit(N_MEMBERS)
        .dropDuplicates(["MBRSHP_SID"])
)

selected_members.createOrReplaceTempView("selected_members")
cohort_sids = F.broadcast(selected_members.select("MBRSHP_SID").distinct())

# -----------------------------
# Column metadata dict and cols_df (SHAPE: table_name, column_name, data_type)
# -----------------------------
cols_df_full = spark.sql(f"""
  SELECT table_schema, table_name, column_name, UPPER(data_type) AS data_type
  FROM {CATALOG}.information_schema.columns
  WHERE table_schema IN ({",".join([f"'{s}'" for s in ALL_SCHEMAS])})
""")

# Schema-less view of columns
cols_df = cols_df_full.select("table_name", "column_name", "data_type")

# Build cols_by_table using full info (schema + table)
grouped_rows = (
    cols_df_full.groupBy("table_schema", "table_name")
                .agg(F.collect_list(F.struct("column_name","data_type")).alias("cols"))
                .collect()
)

cols_by_table = {}
for r in grouped_rows:
    key = (r["table_schema"], r["table_name"])
    cols_by_table[key] = [
        {"column_name": c["column_name"], "data_type": c["data_type"]}
        for c in r["cols"]
    ]

# -----------------------------
# REQUIRES DETAIL  = f"{CATALOG}.{SCHEMA}.transaction_fiscal_detail"
# link_plan_df (SHAPE: table_schema, table_name, strategy, join_key)
# For each table, how do I link it back to a member?
    # strategy = "direct_sid" if table has MBRSHP_SID.
    # strategy = "direct_nbr" if table has MBRSHP_NBR only.
    # strategy = "via_purch_hdr" if it has PURCH_HDR_ID (with detail_fiscal)
    # strategy = "via_purch_dtl" if it has PURCH_DTL_ID.
    # link_plan_df columns:
# table_schema: pe_slv or pe
    # table_name : actual table name
    # strategy : how to reach the member cohort
  # join_key : the key column used in that strategy
# -----------------------------
def build_inferred_link_plan_with_schema():
    rows = []
    has_detail = table_exists(DETAIL)

    for (schema_name, table_name), pairs in cols_by_table.items():
        cols = {p["column_name"].upper() for p in pairs}
        strat, key = None, None

        if "MBRSHP_SID" in cols:
            strat, key = "direct_sid", "MBRSHP_SID"
        elif "MBRSHP_NBR" in cols:
            strat, key = "direct_nbr", "MBRSHP_NBR"
        elif has_detail and "PURCH_HDR_ID" in cols:
            strat, key = "via_purch_hdr", "PURCH_HDR_ID"
        elif has_detail and "PURCH_DTL_ID" in cols:
            strat, key = "via_purch_dtl", "PURCH_DTL_ID"

        if strat:
            rows.append((schema_name, table_name, strat, key))

    return spark.createDataFrame(
        rows,
        schema="table_schema string, table_name string, strategy string, join_key string"
    )

# Generate the link plan 
link_plan_df = build_inferred_link_plan_with_schema()

# ---------- filter link_plan_df to tables that have an S3 mapping ----------
s3_keys = list(S3_FILES.keys())  # logical names available in S3

link_plan_df = link_plan_df.filter(F.col("table_name").isin(s3_keys))

print("Filtered link_plan_df to tables that exist in S3_FILES:")
display(link_plan_df)
# -------------------------------------------------------------------------------

# -----------------------------
# Print dtypes for verification
# -----------------------------
print("selected_members:", selected_members.dtypes)
print("cohort_sids:", cohort_sids.dtypes)
print("cols_df:", cols_df.dtypes)
print("link_plan_df:", link_plan_df.dtypes)

link_plan_df.filter(F.col("strategy").like("via_purch%")).show(truncate=False)


In [0]:

# MBRSHP_SID
# MBRSHP_NBR
# PURCH_HDR_ID (and DETAIL exists)
# PURCH_DTL_ID (and DETAIL exists)

link_plan_df.show(100, truncate=False)


### Analysis

In [0]:
from pyspark.sql import functions as F, types as T

# Type groups
NUMERIC_TYPES = {
    "INT", "INTEGER", "BIGINT", "SMALLINT", "TINYINT",
    "DOUBLE", "FLOAT", "REAL", "DECIMAL", "NUMERIC"
}
DATE_TYPES    = {"DATE", "TIMESTAMP"}
STRING_TYPES  = {"STRING", "CHAR", "VARCHAR"}


def cols_by_type_from_dict(table_name: str):
    """
    Use cols_by_table (built from information_schema) to get
    date, numeric, and string columns for a given table_name.
    """
    date_cols   = []
    num_cols    = []
    string_cols = []

    for (schema_name, tname), cols in cols_by_table.items():
        if tname != table_name:
            continue
        for c in cols:
            col_name = c["column_name"]
            dt       = c["data_type"].upper()
            if dt in DATE_TYPES:
                date_cols.append(col_name)
            elif dt in NUMERIC_TYPES:
                num_cols.append(col_name)
            elif dt in STRING_TYPES:
                string_cols.append(col_name)
        break

    return (
        sorted(set(date_cols)),
        sorted(set(num_cols)),
        sorted(set(string_cols)),
    )


def infer_categorical_cols(df, string_cols, max_cardinality=100):
    """
    Return subset of string_cols that look 'categorical' (low cardinality).
    """
    categorical = []
    for c in string_cols:
        if c not in df.columns:
            continue
        distinct_cnt = (
            df.select(F.col(c))
              .distinct()
              .limit(max_cardinality + 1)
              .count()
        )
        if distinct_cnt <= max_cardinality:
            categorical.append(c)
    return sorted(categorical)


def compute_metrics(df, date_cols, num_cols):
    """
    Compute row_count, date min/max, numeric sum/avg/std for given columns.
    """
    rec = {}
    rec["row_count"] = df.count()

    # Dates
    for c in date_cols:
        if c in df.columns:
            stats = df.select(
                F.min(F.col(c)).alias("min"),
                F.max(F.col(c)).alias("max")
            ).first()
            rec[f"date_min__{c}"] = stats["min"]
            rec[f"date_max__{c}"] = stats["max"]

    # Numerics
    for c in num_cols:
        if c in df.columns:
            stats = df.select(
                F.sum(F.col(c)).alias("sum"),
                F.avg(F.col(c)).alias("avg"),
                F.stddev(F.col(c)).alias("std")
            ).first()
            rec[f"num_sum__{c}"] = float(stats["sum"]) if stats["sum"] is not None else 0.0
            rec[f"num_avg__{c}"] = float(stats["avg"]) if stats["avg"] is not None else 0.0
            rec[f"num_std__{c}"] = float(stats["std"]) if stats["std"] is not None else 0.0

    return rec


def compare_metrics(db_rec, s3_rec, date_cols, num_cols, tol_ratio=0.01, tol_abs=1e-6):
    """
    Compare DB vs S3 metrics for row_count, date min/max, numeric sum/avg/std.
    """
    issues = []

    # Row count
    db_rows = db_rec.get("row_count", 0)
    s3_rows = s3_rec.get("row_count", 0)
    if db_rows != s3_rows:
        issues.append(f"row_count mismatch: db={db_rows}, s3={s3_rows}")

    # Dates
    for c in date_cols:
        db_min = str(db_rec.get(f"date_min__{c}", ""))
        db_max = str(db_rec.get(f"date_max__{c}", ""))
        s3_min = str(s3_rec.get(f"date_min__{c}", ""))
        s3_max = str(s3_rec.get(f"date_max__{c}", ""))
        if db_min != s3_min:
            issues.append(f"{c} min mismatch: db={db_min}, s3={s3_min}")
        if db_max != s3_max:
            issues.append(f"{c} max mismatch: db={db_max}, s3={s3_max}")

    # Numerics
    def close(a, b):
        if a is None and b is None:
            return True
        if a is None or b is None:
            return False
        if abs(a - b) <= tol_abs:
            return True
        denom = max(abs(a), abs(b), tol_abs)
        return abs(a - b) / denom <= tol_ratio

    for c in num_cols:
        for stat in ["sum", "avg", "std"]:
            k = f"num_{stat}__{c}"
            db_v = db_rec.get(k, 0.0)
            s3_v = s3_rec.get(k, 0.0)
            if not close(db_v, s3_v):
                issues.append(f"{c} {stat} mismatch: db={db_v}, s3={s3_v}")

    verdict = "OK" if not issues else "MISMATCH"
    return verdict, issues


def to_maps(rec, date_cols, num_cols):
    """
    Turn flat metric dict into 5 maps: date_min, date_max, sum, avg, std.
    """
    date_min = {}
    date_max = {}
    num_sum  = {}
    num_avg  = {}
    num_std  = {}

    for c in date_cols:
        dmin = rec.get(f"date_min__{c}")
        dmax = rec.get(f"date_max__{c}")
        if dmin is not None:
            date_min[c] = str(dmin)
        if dmax is not None:
            date_max[c] = str(dmax)

    for c in num_cols:
        s  = rec.get(f"num_sum__{c}")
        a  = rec.get(f"num_avg__{c}")
        st = rec.get(f"num_std__{c}")
        if s is not None:
            num_sum[c] = float(s)
        if a is not None:
            num_avg[c] = float(a)
        if st is not None:
            num_std[c] = float(st)

    return date_min, date_max, num_sum, num_avg, num_std




In [0]:
def s3_cohort_df(table_name: str, strategy: str, join_key: str):
    """
    Returns (df, err). df is sliced to the cohort using same join path as DB side.
    Column names are normalized to UPPER to match DB.
    """
    if table_name not in S3_FILES:
        return None, f"S3 path missing for '{table_name}'"
    try:
        base = normalize_cols_upper(read_s3_any(S3_FILES[table_name]))
    except Exception as e:
        return None, f"read error for {table_name}: {e}"

    # Direct MBRSHP_SID
    if strategy == "direct_sid":
        if "MBRSHP_SID" not in base.columns:
            return None, "S3 slice needs MBRSHP_SID but column not present"
        df = base.join(cohort_sids, on="MBRSHP_SID", how="inner")
        return df, None

    # Direct MBRSHP_NBR
    if strategy == "direct_nbr":
        if "master_member" not in S3_FILES:
            return None, "S3 master_member path missing for MBRSHP_NBR bridge"
        s3_member = normalize_cols_upper(read_s3_any(S3_FILES["master_member"]))
        if "MBRSHP_NBR" not in base.columns or "MBRSHP_NBR" not in s3_member.columns:
            return None, "MBRSHP_NBR not present in S3 base/master_member"
        nbr = (
            s3_member
            .select("MBRSHP_SID","MBRSHP_NBR")
            .where(F.col("MBRSHP_NBR").isNotNull())
            .dropDuplicates()
        )
        df = (base.join(F.broadcast(nbr), on="MBRSHP_NBR", how="inner")
                  .join(cohort_sids, on="MBRSHP_SID", how="inner"))
        return df, None

    # via PURCH_HDR_ID
    if strategy == "via_purch_hdr":
        if "transaction_fiscal_detail" not in S3_FILES:
            return None, "S3 transaction_fiscal_detail path missing for PURCH_HDR_ID bridge"
        s3_detail = (
            normalize_cols_upper(read_s3_any(S3_FILES["transaction_fiscal_detail"]))
            .select("PURCH_HDR_ID","MBRSHP_SID")
            .dropDuplicates()
        )
        if "PURCH_HDR_ID" not in base.columns or "PURCH_HDR_ID" not in s3_detail.columns:
            return None, "PURCH_HDR_ID not present in S3 base/transaction_fiscal_detail"
        df = (base.join(F.broadcast(s3_detail), on="PURCH_HDR_ID", how="inner")
                 .join(cohort_sids, on="MBRSHP_SID", how="inner"))
        return df, None

    # via PURCH_DTL_ID
    if strategy == "via_purch_dtl":
        if "transaction_fiscal_detail" not in S3_FILES:
            return None, "S3 transaction_fiscal_detail path missing for PURCH_DTL_ID bridge"
        s3_detail = (
            normalize_cols_upper(read_s3_any(S3_FILES["transaction_fiscal_detail"]))
            .select("PURCH_DTL_ID","MBRSHP_SID")
            .dropDuplicates()
        )
        if "PURCH_DTL_ID" not in base.columns or "PURCH_DTL_ID" not in s3_detail.columns:
            return None, "PURCH_DTL_ID not present in S3 base/transaction_fiscal_detail"
        df = (base.join(F.broadcast(s3_detail), on="PURCH_DTL_ID", how="inner")
                 .join(cohort_sids, on="MBRSHP_SID", how="inner"))
        return df, None

    return None, f"unknown strategy '{strategy}'"
    

In [0]:
print("selected_members:", selected_members.dtypes)
print("link_plan_df:", link_plan_df.dtypes)
print("cols_df_full:", cols_df_full.dtypes)
print("cohort_sids:", cohort_sids.dtypes)


In [0]:
from pyspark.sql import functions as F, types as T

def table_fq(schema_name: str, tname: str) -> str:
    return f"{CATALOG}.{schema_name}.{tname}"

def make_empty_result(schema_name, tname, strat, key, status, issues,
                      db_row_count=0, s3_row_count=0):
    return {
        "table_name": f"{schema_name}.{tname}",
        "strategy": strat,
        "join_key": key,
        "join_path": None,
        "status": status,
        "issues": issues if isinstance(issues, list) else [issues],
        "db_row_count": db_row_count,
        "s3_row_count": s3_row_count,
        "date_cols": [], "numeric_cols": [],
        "string_cols": [], "categorical_cols": [],
        "db_date_min": {}, "db_date_max": {},
        "s3_date_min": {}, "s3_date_max": {},
        "db_sum": {}, "db_avg": {}, "db_std": {},
        "s3_sum": {}, "s3_avg": {}, "s3_std": {},
    }

def build_db_cohort_slice(schema_name, tname, strat, key):
    """
    Build the DB-side cohort slice for a given table + strategy.
    Returns (db_df, join_path, err or None).
    If err is not None, db_df/join_path are undefined.
    """
    fq = table_fq(schema_name, tname)
    if not table_exists(fq):
        return None, None, "SKIPPED_DB_TABLE_MISSING: databricks table missing"

    base_df = spark.table(fq)
    cols_upper = [c.upper() for c in base_df.columns]

    # direct_sid
    if strat == "direct_sid":
        if "MBRSHP_SID" not in cols_upper:
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: MBRSHP_SID missing in DB base table"
        db_df = base_df.join(spark.table("selected_members"),
                             on="MBRSHP_SID", how="inner")
        join_path = f"{schema_name}.{tname}.MBRSHP_SID = selected_members.MBRSHP_SID"
        return db_df, join_path, None

    # direct_nbr
    if strat == "direct_nbr":
        if "master_member" not in S3_FILES:
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: S3 master_member missing; cannot bridge MBRSHP_NBR → MBRSHP_SID"

        s3_member_bridge = normalize_cols_upper(read_s3_any(S3_FILES["master_member"]))
        if "MBRSHP_NBR" not in s3_member_bridge.columns or "MBRSHP_SID" not in s3_member_bridge.columns:
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: MBRSHP_NBR/MBRSHP_SID missing in S3 master_member"

        if "MBRSHP_NBR" not in cols_upper:
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: MBRSHP_NBR missing in DB base table"

        nbr_bridge = (
            s3_member_bridge
            .select("MBRSHP_SID", "MBRSHP_NBR")
            .where(F.col("MBRSHP_NBR").isNotNull())
            .dropDuplicates()
        )

        db_df = (
            base_df
            .join(F.broadcast(nbr_bridge), on="MBRSHP_NBR", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )
        join_path = f"{schema_name}.{tname}.MBRSHP_NBR → S3 master_member.MBRSHP_SID → selected_members"
        return db_df, join_path, None

    # via_purch_hdr
    if strat == "via_purch_hdr":
        if "PURCH_HDR_ID" not in cols_upper or not table_exists(DETAIL):
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: DETAIL bridge missing in DB or PURCH_HDR_ID not in base"

        db_bridge = (
            spark.table(DETAIL)
                 .select("PURCH_HDR_ID", "MBRSHP_SID")
                 .distinct()
        )
        db_df = (
            base_df
            .join(F.broadcast(db_bridge), on="PURCH_HDR_ID", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )
        join_path = f"{schema_name}.{tname}.PURCH_HDR_ID → {DETAIL}.MBRSHP_SID → selected_members"
        return db_df, join_path, None

    # via_purch_dtl
    if strat == "via_purch_dtl":
        if "PURCH_DTL_ID" not in cols_upper or not table_exists(DETAIL):
            return None, None, "SKIPPED_DB_BRIDGE_MISSING: DETAIL bridge missing in DB or PURCH_DTL_ID not in base"

        db_bridge = (
            spark.table(DETAIL)
                 .select("PURCH_DTL_ID", "MBRSHP_SID")
                 .distinct()
        )
        db_df = (
            base_df
            .join(F.broadcast(db_bridge), on="PURCH_DTL_ID", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )
        join_path = f"{schema_name}.{tname}.PURCH_DTL_ID → {DETAIL}.MBRSHP_SID → selected_members"
        return db_df, join_path, None

    return None, None, "SKIPPED_UNKNOWN_STRATEGY: unknown strategy"


rows = []

for r in link_plan_df.orderBy("strategy", "table_schema", "table_name").toLocalIterator():
    schema_name = r["table_schema"]
    tname       = r["table_name"]
    strat       = r["strategy"]
    key         = r["join_key"]

    # 1) Build DB cohort slice
    db_df, join_path, db_err = build_db_cohort_slice(schema_name, tname, strat, key)
    if db_err:
        status, msg = db_err.split(":", 1)
        rows.append(
            make_empty_result(schema_name, tname, strat, key,
                              status=status.strip(), issues=msg.strip(),
                              db_row_count=db_df.count() if db_df is not None else 0)
        )
        continue

    # 2) Build S3 cohort slice
    s3_df, s3_err = s3_cohort_df(tname, strat, key)
    if s3_err:
        rows.append(
            make_empty_result(schema_name, tname, strat, key,
                              status="SKIPPED_S3", issues=s3_err,
                              db_row_count=db_df.count(), s3_row_count=0)
        )
        continue

    # materialize slices for debugging
    safe_name = f"{schema_name}_{tname}".replace(".", "_")
    db_df.createOrReplaceTempView(f"dbg_{safe_name}")
    s3_df.createOrReplaceTempView(f"s3g_{safe_name}")

    # 3) Align & compare metrics
    date_cols, num_cols, string_cols = cols_by_type_from_dict(tname)
    date_cols   = [c for c in date_cols   if c in db_df.columns and c in s3_df.columns]
    num_cols    = [c for c in num_cols    if c in db_df.columns and c in s3_df.columns]
    string_cols = [c for c in string_cols if c in db_df.columns and c in s3_df.columns]

    cat_cols_db = infer_categorical_cols(db_df, string_cols, max_cardinality=100)
    cat_cols_s3 = infer_categorical_cols(s3_df, string_cols, max_cardinality=100)
    cat_cols    = sorted(set(cat_cols_db).intersection(cat_cols_s3))

    db_rec = compute_metrics(db_df, date_cols, num_cols)
    s3_rec = compute_metrics(s3_df, date_cols, num_cols)

    verdict, issues = compare_metrics(db_rec, s3_rec, date_cols, num_cols)

    db_date_min, db_date_max, db_sum, db_avg, db_std = to_maps(db_rec, date_cols, num_cols)
    s3_date_min, s3_date_max, s3_sum, s3_avg, s3_std = to_maps(s3_rec, date_cols, num_cols)

    rows.append({
        "table_name": f"{schema_name}.{tname}",
        "strategy": strat,
        "join_key": key,
        "join_path": join_path,
        "status": verdict,
        "issues": issues,
        "db_row_count": db_rec.get("row_count", 0),
        "s3_row_count": s3_rec.get("row_count", 0),
        "date_cols": date_cols,
        "numeric_cols": num_cols,
        "string_cols": string_cols,
        "categorical_cols": cat_cols,
        "db_date_min": db_date_min, "db_date_max": db_date_max,
        "s3_date_min": s3_date_min, "s3_date_max": s3_date_max,
        "db_sum": db_sum, "db_avg": db_avg, "db_std": db_std,
        "s3_sum": s3_sum, "s3_avg": s3_avg, "s3_std": s3_std,
    })

# Build validation DataFrame
schema = T.StructType([
    T.StructField("table_name", T.StringType(), False),
    T.StructField("strategy", T.StringType(), False),
    T.StructField("join_key", T.StringType(), False),
    T.StructField("join_path", T.StringType(), True),
    T.StructField("status", T.StringType(), False),
    T.StructField("issues", T.ArrayType(T.StringType()), True),
    T.StructField("db_row_count", T.LongType(), False),
    T.StructField("s3_row_count", T.LongType(), False),
    T.StructField("date_cols", T.ArrayType(T.StringType()), True),
    T.StructField("numeric_cols", T.ArrayType(T.StringType()), True),
    T.StructField("string_cols", T.ArrayType(T.StringType()), True),
    T.StructField("categorical_cols", T.ArrayType(T.StringType()), True),
    T.StructField("db_date_min", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("db_date_max", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("s3_date_min", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("s3_date_max", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("db_sum", T.MapType(T.StringType(), T.DoubleType()), True),
    T.StructField("db_avg", T.MapType(T.StringType(), T.DoubleType()), True),
    T.StructField("db_std", T.MapType(T.StringType(), T.DoubleType()), True),
    T.StructField("s3_sum", T.MapType(T.StringType(), T.DoubleType()), True),
    T.StructField("s3_avg", T.MapType(T.StringType(), T.DoubleType()), True),
    T.StructField("s3_std", T.MapType(T.StringType(), T.DoubleType()), True),
])


In [0]:
%sql
SELECT * FROM dbg_pe_slv_master_member LIMIT 1500;



In [0]:
%sql
SELECT * FROM s3g_pe_slv_master_member LIMIT 1500;


In [0]:

validation_table.createOrReplaceTempView("cohort_validation_db_vs_s3")
# Register as a temp view for SQL
validation_table.createOrReplaceTempView("validation_results")

display(validation_table.orderBy(F.asc("status"), F.desc("db_row_count")))


In [0]:
# Register as a temp view for SQL
validation_table.createOrReplaceTempView("validation_results")

In [0]:
%sql
SELECT table_name, db_row_count, s3_row_count, status FROM validation_results;

In [0]:
%sql
SELECT
  table_name,
  db_row_count,
  s3_row_count,
  (s3_row_count - db_row_count) AS diff,
  ROUND(
    CASE WHEN db_row_count = 0 THEN NULL
         ELSE 100.0 * (s3_row_count - db_row_count) / db_row_count
    END, 2
  ) AS pct_diff
FROM validation_results
WHERE db_row_count <> s3_row_count
ORDER BY ABS(pct_diff) DESC NULLS LAST;

In [0]:
%sql
SELECT status, COUNT(*) AS n_tables
FROM validation_results
GROUP BY status
ORDER BY n_tables DESC;

In [0]:

display(
    validation_table
      .where(F.col("status") == "MISMATCH")
      .orderBy(F.desc("db_row_count"))
)


In [0]:
display(
    validation_table
      .where(F.col("status") == "OK")
      .orderBy(F.desc("db_row_count"))
)


In [0]:
display(
    validation_table
      .where(F.array_contains(F.col("issues"), "databricks table missing"))
)


In [0]:
mismatch_counts = (
    validation_table
      .withColumn("row_diff", F.col("db_row_count") - F.col("s3_row_count"))
      .where(F.col("status") == "MISMATCH")
      .select("table_name", "strategy", "db_row_count", "s3_row_count", "row_diff")
      .orderBy(F.desc(F.abs(F.col("row_diff"))))
)

display(mismatch_counts)


In [0]:
status_counts = (
    validation_table
      .groupBy("status")
      .count()
      .orderBy(F.desc("count"))
)

display(status_counts)

In [0]:

spark.sql("""
  SELECT table_name, strategy, status, db_row_count, s3_row_count
  FROM cohort_validation_db_vs_s3
  ORDER BY status, db_row_count DESC
""").show(truncate=False)


In [0]:
from pyspark.sql import functions as F, types as T

# these already exist in your notebook:
# - CATALOG, SCHEMA, S3_FILES
# - table_exists, read_s3_any, normalize_cols_upper
# - selected_members temp view, cohort_sids broadcast
# - link_plan_df: table_schema, table_name, strategy, join_key
# - s3_cohort_df(table_name, strategy, join_key)
# - cols_by_table (keyed by (table_schema, table_name))
# For each such table:
    # Rebuild the DB cohort slice (same strategy).
    # Build the S3 cohort slice via s3_cohort_df.
    # Compute aggregates (row_count, date min/max, numeric sum/avg/std) per side.
    # Return a wide DataFrame agg_df with one row per (table, side) and all metrics.

def numeric_cols_for(schema_name: str, table_name: str):
    pairs = [
        (c["column_name"], str(c["data_type"]).upper())
        for c in cols_by_table.get((schema_name, table_name), [])
    ]
    num_types = {
        "INT", "INTEGER", "BIGINT", "SMALLINT", "TINYINT",
        "DOUBLE", "FLOAT", "REAL", "DECIMAL", "NUMERIC"
    }
    return [c for c, t in pairs if t in num_types]

def date_cols_for(schema_name: str, table_name: str):
    pairs = [
        (c["column_name"], str(c["data_type"]).upper())
        for c in cols_by_table.get((schema_name, table_name), [])
    ]
    date_types = {"DATE", "TIMESTAMP"}
    return [c for c, t in pairs if t in date_types]

def agg_exprs(schema_name: str, table_name: str):
    dcols = date_cols_for(schema_name, table_name)
    ncols = numeric_cols_for(schema_name, table_name)

    aggs = [F.count(F.lit(1)).alias("row_count")]
    for c in dcols:
        aggs += [
            F.min(F.col(c)).alias(f"{c}__min"),
            F.max(F.col(c)).alias(f"{c}__max"),
        ]
    for c in ncols:
        col = F.col(c)
        aggs += [
            F.sum(col).alias(f"{c}__sum"),
            F.avg(col).alias(f"{c}__avg"),
            F.stddev_samp(col).alias(f"{c}__std"),
        ]
    return aggs

def build_db_slice(schema_name: str, tname: str, strat: str, join_key: str):
    fq = f"{CATALOG}.{schema_name}.{tname}"

    base_df = spark.table(fq)

    if strat == "direct_sid":
        if "MBRSHP_SID" not in [c.upper() for c in base_df.columns]:
            return None
        return base_df.join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")

    elif strat == "direct_nbr":
        # Use S3 master_member bridge (same as main code)
        if "master_member" not in S3_FILES:
            return None
        s3_member_bridge = normalize_cols_upper(read_s3_any(S3_FILES["master_member"]))
        if "MBRSHP_NBR" not in s3_member_bridge.columns or "MBRSHP_SID" not in s3_member_bridge.columns:
            return None
        if "MBRSHP_NBR" not in [c.upper() for c in base_df.columns]:
            return None

        nbr_bridge = (
            s3_member_bridge
            .select("MBRSHP_SID", "MBRSHP_NBR")
            .where(F.col("MBRSHP_NBR").isNotNull())
            .dropDuplicates()
        )
        return (
            base_df
            .join(F.broadcast(nbr_bridge), on="MBRSHP_NBR", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )

    elif strat == "via_purch_hdr":
        if not table_exists(f"{CATALOG}.{SCHEMA}.detail_fiscal"):
            return None
        if "PURCH_HDR_ID" not in [c.upper() for c in base_df.columns]:
            return None
        bridge = (
            spark.table(f"{CATALOG}.{SCHEMA}.detail_fiscal")
                 .select("PURCH_HDR_ID","MBRSHP_SID")
                 .distinct()
        )
        return (
            base_df
            .join(F.broadcast(bridge), on="PURCH_HDR_ID", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )

    elif strat == "via_purch_dtl":
        if not table_exists(f"{CATALOG}.{SCHEMA}.detail_fiscal"):
            return None
        if "PURCH_DTL_ID" not in [c.upper() for c in base_df.columns]:
            return None
        bridge = (
            spark.table(f"{CATALOG}.{SCHEMA}.detail_fiscal")
                 .select("PURCH_DTL_ID","MBRSHP_SID")
                 .distinct()
        )
        return (
            base_df
            .join(F.broadcast(bridge), on="PURCH_DTL_ID", how="inner")
            .join(spark.table("selected_members"), on="MBRSHP_SID", how="inner")
        )

    return None

### OK status

In [0]:
display(
  spark.sql("""
    SELECT *
    FROM cohort_validation_db_vs_s3
    WHERE status = 'OK'
    ORDER BY table_name
  """)
)


### Not OK status

In [0]:
display(
  spark.sql("""
    SELECT *
    FROM cohort_validation_db_vs_s3
    WHERE status <> 'OK'
    ORDER BY table_name
  """)
)


In [0]:


# --------------------------------------
# 1) Identify tables with non-OK status
# --------------------------------------
not_ok = spark.sql("""
  SELECT DISTINCT table_name, strategy, join_key
  FROM cohort_validation_db_vs_s3
  WHERE status <> 'OK'
""")

# table_name here is schema.table_name (we wrote it that way)
# so we need to split schema and name
rows = []
for r in not_ok.collect():
    full_name = r["table_name"]        # e.g. "pe_slv.awards"
    strat     = r["strategy"]
    key       = r["join_key"]

    schema_name, tname = full_name.split(".", 1)

    # --- DB side ---
    try:
        db_slice = build_db_slice(schema_name, tname, strat, key)
    except Exception as e:
        rows.append((full_name, strat, "db", None, f"db_slice_error: {e}"))
        continue

    if db_slice is None:
        rows.append((full_name, strat, "db", None, "db_slice_missing"))
    else:
        db_agg = (
            db_slice
            .agg(*agg_exprs(schema_name, tname))
            .withColumn("table_name", F.lit(full_name))
            .withColumn("side", F.lit("db"))
            .withColumn("strategy", F.lit(strat))
        )
        rows.append(db_agg)

    # --- S3 side ---
    try:
        s3_df, s3_err = s3_cohort_df(tname, strat, key)
    except Exception as e:
        rows.append((full_name, strat, "s3", None, f"s3_slice_error: {e}"))
        continue

    if s3_err:
        rows.append((full_name, strat, "s3", None, s3_err))
    else:
        s3_agg = (
            s3_df
            .agg(*agg_exprs(schema_name, tname))
            .withColumn("table_name", F.lit(full_name))
            .withColumn("side", F.lit("s3"))
            .withColumn("strategy", F.lit(strat))
        )
        rows.append(s3_agg)

# --------------------------------------
# 2) Union all aggregation DataFrames
# --------------------------------------
agg_df = None
error_rows = []

for x in rows:
    if isinstance(x, tuple):
        # (table_name, strat, side, None, err_msg)
        error_rows.append(x)
    else:
        agg_df = x if agg_df is None else agg_df.unionByName(x, allowMissingColumns=True)

if agg_df is not None:
    lead = ["table_name", "side", "strategy", "row_count"]
    rest = [c for c in agg_df.columns if c not in lead]
    agg_df = agg_df.select(*lead, *rest)

display(agg_df)


In [0]:
display(
    agg_df
      .select("table_name", "side", "row_count")
      .orderBy("table_name", "side")
)
print("agg_df rows:", agg_df.count())
print("distinct tables in agg_df:", agg_df.select("table_name").distinct().count())

In [0]:
# Convert error_rows (list of tuples) into a Spark DataFrame for inspection
error_schema = T.StructType([
    T.StructField("table_name", T.StringType()),
    T.StructField("strategy",   T.StringType()),
    T.StructField("side",       T.StringType()),
    T.StructField("dummy",      T.StringType()),
    T.StructField("error_msg",  T.StringType()),
])

error_df = spark.createDataFrame(
    [ (t, s, side, None, msg) for (t, s, side, _, msg) in error_rows ],
    schema=error_schema
)

display(error_df.orderBy("table_name", "side"))
print("error_rows count:", len(error_rows))